# nb59 - Physics-template deblending, stage 1: does the mechanism exist? (H23a)

**Error analysis.** The 0.0402 stack's remaining error decomposes as ~0.012 photonic in-window contamination + ~0.005 reconstruction + 0.0235 floor. The contamination is overlapping EM showers - every label-free discriminator fails because pileup photons ARE photons.

**Question.** Can a physics-template fit - window modeled as a sum of N universal EM blobs - estimate the pileup energy per event, with the template measured from OUR clean data (no truth labels anywhere)?

**Hypothesis.** H23a: a 2-blob Grindhammer-form fit (signal blob near the seed + one free pileup blob) recovers per-event pileup energy with corr > 0.5 against the containment-prior estimate of true pileup in the high-contamination tertile. Precedents: Lednev/GAMS chi2 N-shower fits (NIM A366, 1995; template measured from clean electrons); LHCb merged-pi0 splitting (LHCb-2003-091); Grindhammer-Peters profile validity for sampling calorimeters (hep-ex/0001020); BLISS amortized deblending with known PSF (arXiv:2102.02409).

**Proof criterion (pre-registered).** corr(E_pileup_fit, E_pileup_prior) > 0.5 in the high-contamination tertile of the minbias TEST split -> stage 2 (amortized slot head) goes ahead; corr < 0.3 -> H23 closed (template degeneracy wins). Everything measured on clean or computed per event; no training, no tuned knobs.

In [1]:
import os, sys, time, pathlib
import numpy as np, pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'plotly_white'
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, make_windows, splits_for, THRESH
OUT = REPO / 'reports' / 'predictions'
DEVICE = os.environ.get('NB59_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB59_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
print(f'device {DEVICE} | mode {MODE} | build {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events
device cuda | mode full | build 129s


## Measure the universal profile from clean data

Grindhammer 2-component form in pitch units, fit per region on clean windows: cell fraction ~ p*core(Rc) + (1-p)*tail(Rt), blob center = energy centroid. Parameters have physical meaning (core/tail Moliere-like radii); nothing is fit on minbias.

In [2]:
W = 4
def prof2d(dx, dy, Rc, Rt, p):
    r2 = dx ** 2 + dy ** 2
    core = Rc ** 2 / (np.pi * (r2 + Rc ** 2) ** 2)
    tail = Rt ** 2 / (np.pi * (r2 + Rt ** 2) ** 2)
    return p * core + (1 - p) * tail
PROF = {}
from scipy.optimize import minimize as spmin
for reg in range(len(PITCH)):
    pts = []
    for ev in CE:
        if ev['reg'] != reg: continue
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 4: continue
        e = ev['e'][m]; di = ev['di'][m].astype(float); dj = ev['dj'][m].astype(float)
        se = e.sum()
        cx = (e * di).sum() / se; cy = (e * dj).sum() / se
        pts.append(np.stack([di - cx, dj - cy, e / se], 1))
    if len(pts) < 50: continue
    P = np.concatenate(pts)
    def loss(th):
        Rc, Rt, p = np.exp(th[0]), np.exp(th[1]), 1 / (1 + np.exp(-th[2]))
        pred = prof2d(P[:, 0], P[:, 1], Rc, Rt, p)
        pred = pred / max(pred.sum() / len(pts), 1e-9) * 1.0
        return float(((prof2d(P[:, 0], P[:, 1], Rc, Rt, p) - P[:, 2]) ** 2).sum())
    res = spmin(loss, [np.log(0.5), np.log(2.0), 0.0], method='Nelder-Mead',
                options=dict(xatol=1e-4, fatol=1e-8, maxiter=2000))
    Rc, Rt, p = float(np.exp(res.x[0])), float(np.exp(res.x[1])), float(1 / (1 + np.exp(-res.x[2])))
    PROF[reg] = (Rc, Rt, p)
    print(f'region {int(PITCH[reg])}mm: Rc {Rc:.3f} Rt {Rt:.3f} p {p:.3f} pitch-units [{len(pts)} clean events]')
fallback = tuple(np.mean([v for v in PROF.values()], 0)) if PROF else (0.5, 2.0, 0.7)
for reg in range(len(PITCH)):
    if reg not in PROF: PROF[reg] = fallback

region 15mm: Rc 0.613 Rt 0.613 p 0.502 pitch-units [3973 clean events]


region 30mm: Rc 0.604 Rt 0.604 p 0.501 pitch-units [5742 clean events]


region 40mm: Rc 0.618 Rt 0.618 p 0.501 pitch-units [8550 clean events]


region 60mm: Rc 0.597 Rt 0.597 p 0.503 pitch-units [10280 clean events]
region 120mm: Rc 0.579 Rt 0.579 p 0.503 pitch-units [1755 clean events]


## Batched 2-blob fit on the minbias test split (GPU)

Blob 1 (signal): center constrained within +/-1 cell of the seed. Blob 2 (pileup): free. Adam on all windows in parallel; N=1 fit as reference; keep the N=2 solution where it improves chi2 by >5%.

In [3]:
rows, keep = make_windows(W, ME)
ktr, kva, kte = splits_for(keep, len(ME))
sumE = np.array([r[1] for r in rows], np.float32)
Et = np.array([r[3] for r in rows], np.float32)
REG = np.array([r[4] for r in rows], int)
crows, _ = make_windows(W, CE)
creg = np.array([r[4] for r in crows]); cet = np.array([r[3] for r in crows]); cse = np.array([r[1] for r in crows])
ratio = cse / (1000.0 * cet)
cedges = np.quantile(cet, np.linspace(0, 1, 7))
ctab = np.full((len(PITCH), 6), np.nan)
for g in range(len(PITCH)):
    for b in range(6):
        hi = cedges[b+1] + (1e-9 if b == 5 else 0)
        mm = (creg == g) & (cet >= cedges[b]) & (cet < hi)
        if mm.sum() >= 20: ctab[g, b] = np.median(ratio[mm])
ctab = np.where(np.isfinite(ctab), ctab, np.nanmedian(ctab))
ebin = np.clip(np.searchsorted(cedges, Et, side='right') - 1, 0, 5)
Epileup_prior = sumE - ctab[REG, ebin] * 1000.0 * Et
kte_a = np.asarray(kte)
SUB = kte_a if MODE == 'full' else kte_a[:400]
L = (2*W+1)**2
NB = len(SUB)
EG = np.zeros((NB, L), np.float32); MASK = np.zeros((NB, L), np.float32)
DI = np.zeros((NB, L), np.float32); DJ = np.zeros((NB, L), np.float32)
SEEDX = np.zeros(NB, np.float32); SEEDY = np.zeros(NB, np.float32)
RC = np.zeros(NB, np.float32); RT = np.zeros(NB, np.float32); PP = np.zeros(NB, np.float32)
for k, idx in enumerate(SUB):
    tok = rows[idx][0]
    n = tok.shape[0]
    e = np.expm1(tok[:, 0]); di = tok[:, 3]; dj = tok[:, 4]
    EG[k, :n] = e; MASK[k, :n] = 1.0; DI[k, :n] = di; DJ[k, :n] = dj
    s = int(np.argmax(e)); SEEDX[k] = di[s]; SEEDY[k] = dj[s]
    Rc, Rt, p = PROF[REG[idx]]
    RC[k], RT[k], PP[k] = Rc, Rt, p
tE, tM = torch.tensor(EG, device=DEVICE), torch.tensor(MASK, device=DEVICE)
tDI, tDJ = torch.tensor(DI, device=DEVICE), torch.tensor(DJ, device=DEVICE)
tRC = torch.tensor(RC, device=DEVICE)[:, None]; tRT = torch.tensor(RT, device=DEVICE)[:, None]
tP = torch.tensor(PP, device=DEVICE)[:, None]
sE = tE.sum(1, keepdim=True)
def blob(cx, cy, lE):
    dx = tDI - cx; dy = tDJ - cy
    r2 = dx ** 2 + dy ** 2
    core = tRC ** 2 / (np.pi * (r2 + tRC ** 2) ** 2)
    tail = tRT ** 2 / (np.pi * (r2 + tRT ** 2) ** 2)
    return torch.exp(lE) * (tP * core + (1 - tP) * tail)
def fit(nblobs, steps=600):
    torch.manual_seed(0)
    sx = torch.tensor(SEEDX, device=DEVICE)[:, None]
    sy = torch.tensor(SEEDY, device=DEVICE)[:, None]
    d1 = torch.zeros(NB, 2, device=DEVICE, requires_grad=True)
    lE1 = torch.log(sE.squeeze(1) * 0.8 + 1.0).clone().requires_grad_(True)
    params = [d1, lE1]
    if nblobs == 2:
        resid0 = tE - blob(sx, sy, torch.log(sE * 0.8 + 1.0).squeeze(1)[:, None] * torch.ones_like(sx))
        far = torch.argmax((resid0 * tM) * ((tDI - sx) ** 2 + (tDJ - sy) ** 2 > 2).float(), 1)
        p2x = tDI.gather(1, far[:, None]).squeeze(1).clone()
        p2y = tDJ.gather(1, far[:, None]).squeeze(1).clone()
        c2 = torch.stack([p2x, p2y], 1).clone().requires_grad_(True)
        lE2 = torch.log(sE.squeeze(1) * 0.2 + 1.0).clone().requires_grad_(True)
        params += [c2, lE2]
    opt = torch.optim.Adam(params, lr=0.05)
    for it in range(steps):
        opt.zero_grad()
        cx1 = sx + torch.tanh(d1[:, 0:1]); cy1 = sy + torch.tanh(d1[:, 1:2])
        pred = blob(cx1, cy1, lE1[:, None])
        if nblobs == 2:
            pred = pred + blob(c2[:, 0:1], c2[:, 1:2], lE2[:, None])
        chi = (((pred - tE) ** 2) * tM / (tE.abs() + 10.0)).sum(1)
        chi.mean().backward()
        opt.step()
    with torch.no_grad():
        cx1 = sx + torch.tanh(d1[:, 0:1]); cy1 = sy + torch.tanh(d1[:, 1:2])
        pred = blob(cx1, cy1, lE1[:, None])
        E1 = torch.exp(lE1)
        E2 = torch.exp(lE2) if nblobs == 2 else torch.zeros_like(E1)
        if nblobs == 2:
            pred = pred + blob(c2[:, 0:1], c2[:, 1:2], lE2[:, None])
        chi = (((pred - tE) ** 2) * tM / (tE.abs() + 10.0)).sum(1)
    return chi.cpu().numpy(), E1.cpu().numpy(), E2.cpu().numpy()
t1 = time.time()
chi1, E1a, _ = fit(1)
chi2, E1b, E2b = fit(2)
print(f'fits done {time.time()-t1:.0f}s | median chi N=1 {np.median(chi1):.1f} -> N=2 {np.median(chi2):.1f}')
use2 = chi2 < 0.95 * chi1
Epileup_fit = np.where(use2, np.maximum(sumE[SUB] - E1b, 0.0), np.maximum(sumE[SUB] - E1a, 0.0))
print(f'N=2 accepted: {use2.mean():.2f} of windows')

fits done 2s | median chi N=1 13078.4 -> N=2 9215.0
N=2 accepted: 0.92 of windows


## Verdict: does the template fit see the pileup?

Pre-registered: corr > 0.5 in the high-contamination tertile -> stage 2; < 0.3 -> H23 closed.

In [4]:
Pt = Epileup_prior[SUB]
cont = sumE[SUB] / np.maximum(1000.0 * Et[SUB], EPS)
qs = np.quantile(cont, [1/3, 2/3])
E1_fit = np.where(use2, E1b, E1a)
S_prior = ctab[REG[SUB], ebin[SUB]] * 1000.0 * Et[SUB]
def partial_corr(x, yv, z):
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = yv - np.polyval(np.polyfit(z, yv, 1), z)
    return np.corrcoef(rx, ry)[0, 1]
print(f'signal side  corr(E1_fit, signal prior):        {np.corrcoef(E1_fit, S_prior)[0,1]:+.3f}')
print(f'pileup side  PARTIAL corr(fit, prior | sumE):   {partial_corr(Epileup_fit, Pt, sumE[SUB]):+.3f}   <- the untrickable number')
print(f'overall corr(E_pileup_fit, prior): {np.corrcoef(Epileup_fit, Pt)[0,1]:+.3f}')
for gsel, lab in [(cont < qs[0], 'low-cont'), ((cont >= qs[0]) & (cont < qs[1]), 'mid-cont'), (cont >= qs[1], 'HIGH-cont')]:
    cc = np.corrcoef(Epileup_fit[gsel], Pt[gsel])[0, 1]
    print(f'{lab:10s}: corr {cc:+.3f} (n={gsel.sum()})')
fig = go.Figure()
sel = np.random.default_rng(0).choice(len(SUB), size=min(3000, len(SUB)), replace=False)
fig.add_scatter(x=Pt[sel], y=Epileup_fit[sel], mode='markers', marker=dict(size=3, opacity=0.4))
fig.add_scatter(x=[0, np.quantile(Pt, 0.99)], y=[0, np.quantile(Pt, 0.99)], mode='lines', line=dict(dash='dot', color='black'))
fig.update_layout(height=450, title='template-fit pileup estimate vs containment-prior truth proxy',
                  xaxis_title='E_pileup prior [MeV]', yaxis_title='E_pileup fit [MeV]')
fig.write_html(REPO / 'reports' / 'figures' / 'interactive' / 'ifig11_deblend_diag.html', include_plotlyjs='cdn')
fig.show()

signal side  corr(E1_fit, signal prior):        +0.810
pileup side  PARTIAL corr(fit, prior | sumE):   +0.637   <- the untrickable number
overall corr(E_pileup_fit, prior): +0.940
low-cont  : corr +0.614 (n=3628)
mid-cont  : corr +0.900 (n=3628)
HIGH-cont : corr +0.929 (n=3628)
